# 2주차 4교시 · 30일 활동 기록을 회원 한 명당 한 행으로 요약하기

**학생용 복습완성본**

오늘의 질문: **회원마다 30줄씩 있는 활동 기록을 어떻게 한 줄로 줄일까요?**

오늘의 완성 결과: 30,000행의 하루 기록을 회원별로 요약하고 기본정보를 붙여 1,000행×16열 표를 저장합니다.

학생이 막히면 정답 전체를 바로 보여주지 말고, [자료구조] → [입력] → [한 줄씩 읽기] → [정상 결과] 순으로 힌트를 줍니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
OUTPUT_DIR = WEEK2_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
frames = []
for file_path in batch_files:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)


## Q1. 문자 숫자와 날짜를 계산할 수 있는 새 열로 바꿉니다

**먼저 예상하기**

- 원본 열을 남기고 계산용 새 열을 만드는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** '미측정' 문자열과 날짜 문자열을 그대로 합산하거나 빼지 않도록 준비합니다.

**현재 자료구조** stay_time_min_num은 float Series, activity_dt와 active_dt는 datetime Series입니다.

**코드 읽기 도움** where는 logged_in이 True인 행에만 날짜를 남깁니다. 그래서 마지막 로그인 날짜를 구할 수 있습니다.

**정상 결과** 12

**오류가 나면** ValueError 또는 object dtype 유지. errors='coerce'와 새 열 이름을 확인합니다.


In [ ]:
activity["activity_dt"] = pd.to_datetime(activity["activity_date"], errors="coerce")
activity["stay_time_min_num"] = pd.to_numeric(activity["stay_time_min"], errors="coerce")
activity["active_dt"] = activity["activity_dt"].where(activity["logged_in"])
print(activity["stay_time_min_num"].isna().sum())

### 강사 설명 메모

- 학생에게 물을 질문: 원본 열을 남기고 계산용 새 열을 만드는 이유는 무엇일까요?
- 기대 답: 원문과 바꾼 결과를 나란히 비교할 수 있고 문제가 생기면 원래 값으로 돌아갈 수 있습니다.
- 그대로 말할 문장: “where는 logged_in이 True인 행에만 날짜를 남깁니다. 그래서 마지막 로그인 날짜를 구할 수 있습니다.”
- 결과 확인: 12
- 복구 순서: errors='coerce'와 새 열 이름을 확인합니다.


## Q2. 같은 회원의 30일 기록을 합계·평균·최근 날짜로 요약합니다

**먼저 예상하기**

- 한 회원에게 30줄씩 있는 기록을 한 줄로 만들려면 무엇을 기준으로 묶어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 하루별 로그를 회원별 지표로 바꾸어 모델과 API가 사용하기 쉽게 만듭니다.

**현재 자료구조** member_activity는 member_id별 한 행인 1,000행×9열 DataFrame입니다.

**코드 읽기 도움** True는 합계에서 1, False는 0처럼 계산되므로 logged_in의 sum은 로그인한 날 수가 됩니다.

**정상 결과** (1000, 9)

**오류가 나면** 한 회원이 여러 행으로 남거나 MultiIndex가 생김. groupby 기준이 member_id인지, 끝에 reset_index()가 있는지 확인합니다.


In [ ]:
member_activity = activity.groupby("member_id").agg(
    active_days=("logged_in", "sum"),
    total_app_opens=("app_opens", "sum"),
    total_stay_time_min=("stay_time_min_num", "sum"),
    total_profile_views=("profile_views", "sum"),
    total_recommendations_viewed=("recommendations_viewed", "sum"),
    total_messages_sent=("messages_sent", "sum"),
    avg_reply_delay_sec=("avg_reply_delay_sec", "mean"),
    last_active_date=("active_dt", "max"),
).reset_index()
print(member_activity.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 한 회원에게 30줄씩 있는 기록을 한 줄로 만들려면 무엇을 기준으로 묶어야 할까요?
- 기대 답: member_id가 같은 행끼리 묶습니다.
- 그대로 말할 문장: “True는 합계에서 1, False는 0처럼 계산되므로 logged_in의 sum은 로그인한 날 수가 됩니다.”
- 결과 확인: (1000, 9)
- 복구 순서: groupby 기준이 member_id인지, 끝에 reset_index()가 있는지 확인합니다.


## Q3. 마지막 로그인 후 며칠이 지났는지 계산합니다

**먼저 예상하기**

- 날짜 자체보다 경과일 숫자가 필요한 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 날짜 자체보다 '마지막 활동 후 며칠 지났는가'를 계산에 바로 쓸 수 있게 만듭니다.

**현재 자료구조** days_since_last_activity는 숫자 Series이며 로그인 기록이 전혀 없는 20명은 NaN입니다.

**코드 읽기 도움** 날짜를 빼면 시간 차이 자료형이 되고, .dt.days가 그 차이에서 일수만 꺼냅니다.

**정상 결과** 20

**오류가 나면** 음수 또는 AttributeError. 뺄셈 순서와 last_active_date dtype을 확인합니다.


In [ ]:
reference_date = pd.Timestamp("2026-07-01")
member_activity["days_since_last_activity"] = (
    reference_date - member_activity["last_active_date"]
).dt.days
print(member_activity["days_since_last_activity"].isna().sum())

### 강사 설명 메모

- 학생에게 물을 질문: 날짜 자체보다 경과일 숫자가 필요한 이유는 무엇일까요?
- 기대 답: 최근 활동 여부를 비교하거나 계산하려면 날짜보다 경과일 숫자가 사용하기 쉽습니다.
- 그대로 말할 문장: “날짜를 빼면 시간 차이 자료형이 되고, .dt.days가 그 차이에서 일수만 꺼냅니다.”
- 결과 확인: 20
- 복구 순서: 뺄셈 순서와 last_active_date dtype을 확인합니다.


## Q4. 회원 기본정보와 활동 요약을 member_id로 붙입니다

**먼저 예상하기**

- 활동 기록이 없는 회원도 남기려면 어느 표를 왼쪽에 두어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 나이·지역 같은 회원 정보와 30일 활동 지표를 같은 행에서 사용합니다.

**현재 자료구조** members는 1,000행×59열, model_table은 필요한 회원 열과 활동 요약을 합친 1,000행×16열 DataFrame입니다.

**코드 읽기 도움** validate='one_to_one'은 양쪽 모두 member_id 한 개당 한 행이어야 한다는 안전 검사입니다.

**정상 결과** (1000, 16)

**오류가 나면** MergeError 또는 행 수가 1,000보다 많음. 두 표의 member_id 중복 수를 먼저 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "members.csv")
member_columns = ["member_id", "gender", "age", "residence_region", "membership_tier", "signup_channel", "meeting_scheduled"]
model_table = members[member_columns].merge(
    member_activity, on="member_id", how="left", validate="one_to_one"
)
print(model_table.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 활동 기록이 없는 회원도 남기려면 어느 표를 왼쪽에 두어야 할까요?
- 기대 답: 회원 1,000명을 모두 남기려면 members를 왼쪽에 둡니다.
- 그대로 말할 문장: “validate='one_to_one'은 양쪽 모두 member_id 한 개당 한 행이어야 한다는 안전 검사입니다.”
- 결과 확인: (1000, 16)
- 복구 순서: 두 표의 member_id 중복 수를 먼저 확인합니다.


## Q5. 회원 한 명당 한 행인지 검사하고 요약표를 저장합니다

**먼저 예상하기**

- 다음 교시가 믿고 읽을 수 있도록 저장 전에 무엇을 검사해야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 다음 교시가 같은 검증된 입력표에서 시작하도록 중간 결과를 고정합니다.

**현재 자료구조** 저장 파일은 1,000행×16열 CSV이며 member_id는 중복되지 않습니다.

**코드 읽기 도움** 중간 결과를 저장하면 다음 교시에서 30,000행 집계를 반복하지 않고도 시작할 수 있습니다.

**정상 결과** member_activity_summary.csv와 (1000, 16)

**오류가 나면** AssertionError 또는 PermissionError. 중복 member_id를 출력하고, 저장 파일이 열려 있으면 닫습니다.


In [ ]:
assert model_table["member_id"].is_unique
assert model_table.shape == (1000, 16)
summary_path = OUTPUT_DIR / "member_activity_summary.csv"
model_table.to_csv(summary_path, index=False)
print(summary_path, model_table.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 다음 교시가 믿고 읽을 수 있도록 저장 전에 무엇을 검사해야 할까요?
- 기대 답: member_id가 모두 한 번씩인지, 표가 (1000, 16)인지 확인합니다.
- 그대로 말할 문장: “중간 결과를 저장하면 다음 교시에서 30,000행 집계를 반복하지 않고도 시작할 수 있습니다.”
- 결과 확인: member_activity_summary.csv와 (1000, 16)
- 복구 순서: 중복 member_id를 출력하고, 저장 파일이 열려 있으면 닫습니다.


## 마무리 확인

- [ ] 오늘 결과가 **30,000행의 하루 기록을 회원별로 요약하고 기본정보를 붙여 1,000행×16열 표를 저장합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
